Movie Ratings With Name

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StructField, StructType, IntegerType, LongType, StringType

def loadMovieNames() -> dict[int,str]:
    movieNames = {}

    with open('../u.item', 'r', encoding='ISO-8859-1', errors='ignore') as f:
        for line in f:
            fields = line.split('|')
            movieNames[int(fields[0])] = fields[1]
        return movieNames
    

spark = SparkSession.builder.appName('PopularMovies').getOrCreate()
spark.sparkContext.setLogLevel("WARN")

nameDict = spark.sparkContext.broadcast(loadMovieNames())

schema = StructType([ \
    StructField('userID', IntegerType()), \
    StructField('movieID', IntegerType()), \
    StructField('rating', IntegerType()), \
    StructField('timestamp', LongType())
    ])

movieRatingsDF = spark.read.option("sep", "\t").schema(schema).csv('../u.data')

movieRatingCounts = movieRatingsDF.groupBy('movieID').count()

@udf
def lookupName(movieID: int) -> str:
    return nameDict.value.get(movieID, 'Unknown')

movieCountWithNames = movieRatingCounts.withColumn('movieTitle', lookupName(F.col('movieID')))
sortedMovieCountsWithNames = movieCountWithNames.orderBy(F.desc('count'))

sortedMovieCountsWithNames.show(10, False)

spark.stop()


+-------+-----+-----------------------------+
|movieID|count|movieTitle                   |
+-------+-----+-----------------------------+
|50     |583  |Star Wars (1977)             |
|258    |509  |Contact (1997)               |
|100    |508  |Fargo (1996)                 |
|181    |507  |Return of the Jedi (1983)    |
|294    |485  |Liar Liar (1997)             |
|286    |481  |English Patient, The (1996)  |
|288    |478  |Scream (1996)                |
|1      |452  |Toy Story (1995)             |
|300    |431  |Air Force One (1997)         |
|121    |429  |Independence Day (ID4) (1996)|
+-------+-----+-----------------------------+
only showing top 10 rows
